### Clustering (DBSCAN)

In [1]:
import numpy as np
import pandas as pd
import optuna
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

In [2]:
# Load your data
data = pd.read_csv('merged_df.csv')

In [3]:
house_df = data[data['Category'] == 'house']
condo_df = data[data['Category'] == 'condo']
apt_df = data[data['Category'] == 'apartment']
com_df = data[data['Category'] == 'commercial']
land_df = data[data['Category'] == 'land']

In [4]:
X_house = house_df[['Price','Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 'LRTHubDist', 'BusCount', 'MallCount', 
                    'SchoolCount', 'HospitalCount', 'ATMCount', 'employment_generation', 'local_economy_growth', 
                    'local_economy_size', 'presence_of_business_and_professional_organizations', 'safety_compliant_business']]
X_condo = condo_df[['Price','Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 'LRTHubDist', 'BusCount', 'MallCount', 
                    'SchoolCount', 'HospitalCount', 'ATMCount', 'employment_generation', 'local_economy_growth', 
                    'local_economy_size', 'presence_of_business_and_professional_organizations', 'safety_compliant_business']]
X_apt = apt_df[['Price','Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 'LRTHubDist', 'BusCount', 'MallCount', 
                    'SchoolCount', 'HospitalCount', 'ATMCount', 'employment_generation', 'local_economy_growth', 
                    'local_economy_size', 'presence_of_business_and_professional_organizations', 'safety_compliant_business']]
X_com = com_df[['Price','Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 'LRTHubDist', 'BusCount', 'MallCount', 
                    'SchoolCount', 'HospitalCount', 'ATMCount', 'employment_generation', 'local_economy_growth', 
                    'local_economy_size', 'presence_of_business_and_professional_organizations', 'safety_compliant_business']]
X_land = land_df[['Price','Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 'LRTHubDist', 'BusCount', 'MallCount', 
                    'SchoolCount', 'HospitalCount', 'ATMCount', 'employment_generation', 'local_economy_growth', 
                    'local_economy_size', 'presence_of_business_and_professional_organizations', 'safety_compliant_business']]

### House

In [5]:
def objective(trial, X):
    """
    Objective function for Optuna to optimize DBSCAN hyperparameters using silhouette score.
    
    Parameters:
        trial: Optuna trial object
        X: Feature matrix
    
    Returns:
        Silhouette score (higher is better)
    """
    # Suggest hyperparameters
    eps = trial.suggest_float("eps", 0.1, 5.0, log=True)  # Search range for neighborhood radius
    min_samples = trial.suggest_int("min_samples", 2, 20)  # Minimum points in a cluster
    
    # Create DBSCAN pipeline
    dbscan_pipeline = make_pipeline(StandardScaler(), DBSCAN(eps=eps, min_samples=min_samples))
    
    # Fit DBSCAN model
    labels = dbscan_pipeline.fit_predict(X)
    
    # Silhouette score requires at least 2 clusters
    num_clusters = len(set(labels) - {-1})  # Exclude noise (-1)
    if num_clusters < 2:
        return -1  # Return a bad score if only one cluster is found

    # Compute silhouette score
    score = silhouette_score(X, labels)
    return score

def tune_dbscan(X, n_trials=50):
    """
    Uses Optuna to find the best DBSCAN hyperparameters.
    
    Parameters:
        X: Feature matrix
        n_trials: Number of optimization trials (default: 50)
    
    Returns:
        Best parameters and corresponding DBSCAN pipeline
    """
    study = optuna.create_study(direction="maximize")  # Maximize silhouette score
    study.optimize(lambda trial: objective(trial, X), n_trials=n_trials)
    
    best_params = study.best_params
    print("Best Hyperparameters:", best_params)
    
    # Train final DBSCAN model with best parameters
    best_dbscan_pipeline = make_pipeline(StandardScaler(), DBSCAN(**best_params))
    best_dbscan_pipeline.fit(X)

    return best_dbscan_pipeline, best_params

In [6]:
# Example usage:
best_dbscan, best_dbscan_params = tune_dbscan(X_house)

[I 2025-03-06 16:15:46,089] A new study created in memory with name: no-name-03713316-91e6-4791-851d-cb0f53e2238c
[I 2025-03-06 16:16:05,340] Trial 0 finished with value: -0.7981234828836986 and parameters: {'eps': 0.12610950682581146, 'min_samples': 10}. Best is trial 0 with value: -0.7981234828836986.
[I 2025-03-06 16:16:22,879] Trial 1 finished with value: -0.7902277292288822 and parameters: {'eps': 0.16833207949097068, 'min_samples': 11}. Best is trial 1 with value: -0.7902277292288822.
[I 2025-03-06 16:16:38,962] Trial 2 finished with value: -0.8434435660433723 and parameters: {'eps': 0.1448977773641353, 'min_samples': 5}. Best is trial 1 with value: -0.7902277292288822.
[I 2025-03-06 16:17:06,376] Trial 3 finished with value: -0.6139006207599499 and parameters: {'eps': 2.2576957909765536, 'min_samples': 12}. Best is trial 3 with value: -0.6139006207599499.
[I 2025-03-06 16:17:21,408] Trial 4 finished with value: -0.7854229715577805 and parameters: {'eps': 0.4220266512982833, 'min

Best Hyperparameters: {'eps': 3.894257000189785, 'min_samples': 20}


In [7]:
def get_top_features_and_silhouette(X, best_dbscan):
    """
    Extracts the top 5 distinguishing features for each cluster 
    and computes the silhouette score.
    
    Parameters:
        X: Feature matrix (DataFrame or NumPy array)
        best_dbscan: Trained DBSCAN model

    Returns:
        top_features_df: DataFrame of top 5 features per cluster
        best_silhouette: Silhouette score of the best clustering
    """
    # Get cluster labels
    labels = best_dbscan.fit_predict(X)

    # Compute silhouette score (only if there are at least 2 clusters)
    num_clusters = len(set(labels) - {-1})  # Exclude noise (-1)
    best_silhouette = silhouette_score(X, labels) if num_clusters > 1 else -1

    # Convert to DataFrame if needed and append cluster labels
    X_df = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X.copy()
    X_df["Cluster"] = labels  
    clustered_data = X_df[X_df["Cluster"] != -1]  # Remove noise

    top_features_dict = {}

    for cluster in np.unique(clustered_data["Cluster"]):
        cluster_data = clustered_data[clustered_data["Cluster"] == cluster].drop(columns=["Cluster"])
        
        feature_means = cluster_data.mean()
        top_features = feature_means.abs().nlargest(5).index.tolist()
        top_values = feature_means[top_features].values.tolist()

        top_features_dict[f"Cluster {cluster}"] = list(zip(top_features, top_values))

    return pd.DataFrame(top_features_dict), best_silhouette

In [8]:
# Example Usage
top_features_dbscan, best_silhouette = get_top_features_and_silhouette(X_house, best_dbscan)

print(f"Best Silhouette Score: {best_silhouette}")
print(top_features_dbscan)

Best Silhouette Score: -0.06254392280063463
                           Cluster 0                         Cluster 1
0         (Price, 48807651.35237454)        (Price, 51900886.33415233)
1    (Floor_Area, 277.2062516798838)   (Floor_Area, 307.3101474201474)
2    (Land_Area, 215.13592408741445)   (Land_Area, 166.57742014742016)
3  (SchoolCount, 10.680035111084207)  (SchoolCount, 20.67076167076167)
4      (BusCount, 9.833464495429505)    (ATMCount, 13.904176904176904)


In [9]:
# Save results into a copy of X_house
X_dbscan = X_house.copy()
X_dbscan["DBSCAN_Cluster"] = best_dbscan.fit_predict(X_house)

# Save as CSV
X_dbscan.to_csv("house_tuned_dbscan.csv", index=False)

### Condo

In [10]:
# Example usage:
best_dbscan, best_dbscan_params = tune_dbscan(X_condo)

[I 2025-03-06 17:05:12,696] A new study created in memory with name: no-name-784ebe79-c4d9-46b4-809a-cead8dea3783
[I 2025-03-06 17:05:32,421] Trial 0 finished with value: -0.5616047641616552 and parameters: {'eps': 2.398520258509104, 'min_samples': 6}. Best is trial 0 with value: -0.5616047641616552.
[I 2025-03-06 17:05:42,869] Trial 1 finished with value: -0.7277794002229144 and parameters: {'eps': 1.1659636177535058, 'min_samples': 6}. Best is trial 0 with value: -0.5616047641616552.
[I 2025-03-06 17:06:26,251] Trial 2 finished with value: -0.7616607935411279 and parameters: {'eps': 3.72277578786764, 'min_samples': 3}. Best is trial 0 with value: -0.5616047641616552.
[I 2025-03-06 17:06:36,074] Trial 3 finished with value: -0.6832777223876877 and parameters: {'eps': 1.019016013771395, 'min_samples': 18}. Best is trial 0 with value: -0.5616047641616552.
[I 2025-03-06 17:06:44,726] Trial 4 finished with value: -0.8232580292489835 and parameters: {'eps': 0.1375332877739883, 'min_samples

Best Hyperparameters: {'eps': 3.24469540223856, 'min_samples': 11}


In [11]:
# Example Usage
top_features_dbscan, best_silhouette = get_top_features_and_silhouette(X_condo, best_dbscan)

print(f"Best Silhouette Score: {best_silhouette}")
print(top_features_dbscan)

Best Silhouette Score: -0.20598374816211557
                         Cluster 0                          Cluster 1
0      (Price, 22752929.631287154)         (Price, 13600122.15568528)
1  (Floor_Area, 82.10573060813533)    (Floor_Area, 61.45557106598985)
2    (BusCount, 17.37378171566653)      (ATMCount, 32.28426395939086)
3   (ATMCount, 17.152637937978252)  (SchoolCount, 12.549492385786802)
4  (SchoolCount, 9.83765606121627)     (BusCount, 10.728003384094755)


In [12]:
# Save results into a copy of X_house
X_dbscan = X_condo.copy()
X_dbscan["DBSCAN_Cluster"] = best_dbscan.fit_predict(X_condo)

# Save as CSV
X_dbscan.to_csv("condo_tuned_dbscan.csv", index=False)

### Apt

In [13]:
# Example usage:
best_dbscan, best_dbscan_params = tune_dbscan(X_apt)

[I 2025-03-06 17:26:07,034] A new study created in memory with name: no-name-2a7f8c6e-c1d7-49b4-81b8-b5c7a6c8c5e1
[I 2025-03-06 17:26:07,068] Trial 0 finished with value: -1.0 and parameters: {'eps': 3.6028173002145936, 'min_samples': 13}. Best is trial 0 with value: -1.0.
[I 2025-03-06 17:26:07,086] Trial 1 finished with value: -1.0 and parameters: {'eps': 4.425867092022887, 'min_samples': 9}. Best is trial 0 with value: -1.0.
[I 2025-03-06 17:26:07,099] Trial 2 finished with value: -0.23039770973756776 and parameters: {'eps': 2.517238691978661, 'min_samples': 20}. Best is trial 2 with value: -0.23039770973756776.
[I 2025-03-06 17:26:07,115] Trial 3 finished with value: -1.0 and parameters: {'eps': 0.3451039486094137, 'min_samples': 17}. Best is trial 2 with value: -0.23039770973756776.
[I 2025-03-06 17:26:07,115] Trial 4 finished with value: -1.0 and parameters: {'eps': 0.105173931747103, 'min_samples': 10}. Best is trial 2 with value: -0.23039770973756776.
[I 2025-03-06 17:26:07,131

Best Hyperparameters: {'eps': 2.8730889776362196, 'min_samples': 9}


In [14]:
# Example Usage
top_features_dbscan, best_silhouette = get_top_features_and_silhouette(X_apt, best_dbscan)

print(f"Best Silhouette Score: {best_silhouette}")
print(top_features_dbscan)

Best Silhouette Score: 0.05937124964030582
                            Cluster 0            Cluster 1
0                (Price, 51958711.25)  (Price, 67610000.0)
1    (Floor_Area, 472.94873076923074)  (Floor_Area, 630.2)
2       (BusCount, 16.53846153846154)  (SchoolCount, 21.9)
3   (SchoolCount, 13.180769230769231)     (ATMCount, 10.6)
4  (Num_Bedrooms, 10.903846153846153)  (Num_Bedrooms, 9.8)


In [15]:
# Save results into a copy of X_house
X_dbscan = X_apt.copy()
X_dbscan["DBSCAN_Cluster"] = best_dbscan.fit_predict(X_apt)

# Save as CSV
X_dbscan.to_csv("apt_tuned_dbscan.csv", index=False)

### Commercial Real Estate

In [16]:
# Example usage:
best_dbscan, best_dbscan_params = tune_dbscan(X_com)

[I 2025-03-06 17:26:08,063] A new study created in memory with name: no-name-b75ec2ae-1448-4863-b9bb-49c024a8214a
[I 2025-03-06 17:26:08,231] Trial 0 finished with value: -0.7704675387339457 and parameters: {'eps': 1.28543330668589, 'min_samples': 4}. Best is trial 0 with value: -0.7704675387339457.
[I 2025-03-06 17:26:08,359] Trial 1 finished with value: -1.0 and parameters: {'eps': 4.626207704216292, 'min_samples': 18}. Best is trial 0 with value: -0.7704675387339457.
[I 2025-03-06 17:26:08,500] Trial 2 finished with value: -0.763149610043525 and parameters: {'eps': 0.13022479420429348, 'min_samples': 2}. Best is trial 2 with value: -0.763149610043525.
[I 2025-03-06 17:26:08,672] Trial 3 finished with value: -0.4652881731985135 and parameters: {'eps': 1.9881331619669822, 'min_samples': 9}. Best is trial 3 with value: -0.4652881731985135.
[I 2025-03-06 17:26:08,845] Trial 4 finished with value: -0.35876262897457095 and parameters: {'eps': 2.1020093712442063, 'min_samples': 7}. Best is

Best Hyperparameters: {'eps': 3.9824068960599446, 'min_samples': 4}


In [17]:
# Example Usage
top_features_dbscan, best_silhouette = get_top_features_and_silhouette(X_com, best_dbscan)

print(f"Best Silhouette Score: {best_silhouette}")
print(top_features_dbscan)

Best Silhouette Score: 0.8538888275947706
                          Cluster 0                         Cluster 1
0       (Price, 206943090.29528704)             (Price, 2413925000.0)
1  (Floor_Area, 1221.0920247507736)              (Land_Area, 49528.5)
2    (Land_Area, 839.3460158129942)              (Floor_Area, 7500.0)
3    (BusCount, 18.247507734616708)  (LRTHubDist, 21.351381824719798)
4    (ATMCount, 17.511172224132004)                  (BusCount, 14.0)


In [18]:
# Save results into a copy of X_house
X_dbscan = X_com.copy()
X_dbscan["DBSCAN_Cluster"] = best_dbscan.fit_predict(X_com)

# Save as CSV
X_dbscan.to_csv("com_tuned_dbscan.csv", index=False)

### Land

In [19]:
# Example usage:
best_dbscan, best_dbscan_params = tune_dbscan(X_land)

[I 2025-03-06 17:26:17,442] A new study created in memory with name: no-name-f3c33dfe-1be8-4f47-b716-e0461b304036
[I 2025-03-06 17:26:18,610] Trial 0 finished with value: -0.4151853270563301 and parameters: {'eps': 1.9942821233133439, 'min_samples': 11}. Best is trial 0 with value: -0.4151853270563301.
[I 2025-03-06 17:26:19,580] Trial 1 finished with value: -0.8209994915507749 and parameters: {'eps': 0.4567945469355818, 'min_samples': 8}. Best is trial 0 with value: -0.4151853270563301.
[I 2025-03-06 17:26:20,567] Trial 2 finished with value: -0.8058161226407552 and parameters: {'eps': 0.2792583405662267, 'min_samples': 8}. Best is trial 0 with value: -0.4151853270563301.
[I 2025-03-06 17:26:21,585] Trial 3 finished with value: -0.7720598836383679 and parameters: {'eps': 0.8309231201643892, 'min_samples': 15}. Best is trial 0 with value: -0.4151853270563301.
[I 2025-03-06 17:26:22,618] Trial 4 finished with value: -0.729785992313693 and parameters: {'eps': 0.9709568275050025, 'min_sam

Best Hyperparameters: {'eps': 2.6288635693115285, 'min_samples': 11}


In [20]:
# Example Usage
top_features_dbscan, best_silhouette = get_top_features_and_silhouette(X_land, best_dbscan)

print(f"Best Silhouette Score: {best_silhouette}")
print(top_features_dbscan)

Best Silhouette Score: -0.11151012808998603
                          Cluster 0                          Cluster 1  \
0       (Price, 180631339.60837334)        (Price, 178492151.91358024)   
1     (Land_Area, 4174.02143480506)     (Land_Area, 767.0296296296297)   
2   (LRTHubDist, 13.86752912286616)    (Floor_Area, 42.34567901234568)   
3     (BusCount, 7.689775367931836)  (SchoolCount, 18.160493827160494)   
4  (SchoolCount, 6.569584301575007)     (ATMCount, 14.024691358024691)   

                           Cluster 2                        Cluster 3  
0        (Price, 1032997072.5862069)             (Price, 722555600.0)  
1     (Land_Area, 3512.974568965517)  (Land_Area, 2699.8866666666663)  
2      (BusCount, 34.01724137931034)                 (BusCount, 45.0)  
3  (SchoolCount, 13.086206896551724)              (SchoolCount, 16.0)  
4      (ATMCount, 9.258620689655173)                  (ATMCount, 9.0)  


In [21]:
# Save results into a copy of X_house
X_dbscan = X_land.copy()
X_dbscan["DBSCAN_Cluster"] = best_dbscan.fit_predict(X_land)

# Save as CSV
X_dbscan.to_csv("land_tuned_dbscan.csv", index=False)